In [6]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler
from scipy.stats import entropy
from scipy.signal import periodogram
import joblib

In [7]:
#pipeline configuration
MODEL_PATH = "iso_forest_nitrogen.pkl"
SCALER_PATH = "iso_scaler.pkl"

In [8]:
# Config
DATA_FOLDER = "PW_0.5h"
TARGET = DATA_FOLDER
TRAIN_PATH = f"{DATA_FOLDER}/train.csv"
VAL_PATH = f"{DATA_FOLDER}/val.csv"
TEST_PATH = f"{DATA_FOLDER}/test.csv"

SENSORS = ["CMS air pressure", "Concentrazione ossigeno base", "Nitrogen pressure", "Oxygen over threshold"]

In [9]:
#helper functions
def spectral_entropy(x):
    """Compute spectral entropy using FFT."""
    freqs, psd = periodogram(x, scaling='spectrum')
    psd_norm = psd / np.sum(psd)
    return entropy(psd_norm)

def derivative_features(x):
    """Compute derivative-based temporal features."""
    dx = np.diff(x)
    return {
        "mean_derivative": np.mean(np.abs(dx)),
        "max_derivative": np.max(np.abs(dx)),
        "num_sign_changes": np.sum(np.diff(np.sign(dx)) != 0),
        "flat_segments": np.sum(dx == 0),
        "longest_flat_segment": longest_run(dx == 0)
    }

def longest_run(arr):
    """Longest consecutive True values."""
    max_run = run = 0
    for val in arr:
        if val:
            run += 1
            max_run = max(max_run, run)
        else:
            run = 0
    return max_run

def out_of_range_fraction(x, min_val, max_val):
    return np.mean((x < min_val) | (x > max_val))

def extreme_jump_fraction(x, threshold):
    dx = np.abs(np.diff(x))
    return np.mean(dx > threshold)

#Feature engineering
def extract_features(df, ranges, jump_thresholds, baseline_stats=None):
    """
    df: RW dataframe (rows = timestamps, columns = sensors)
    ranges: dict {sensor: (min_val, max_val)}
    jump_thresholds: dict {sensor: max_allowed_jump}
    baseline_stats: dict {sensor: (mean, std)} for drift detection
    """
    
    features = {}
    
    # -----------------------------------------------------
    # RW-level missingness indicators
    # -----------------------------------------------------
    missing_fraction_per_sensor = df.isna().mean()
    features["rw_missing_fraction"] = missing_fraction_per_sensor.mean()
    features["rw_num_high_missing_sensors"] = (missing_fraction_per_sensor > 0.2).sum()
    
    # -----------------------------------------------------
    # 1. Per-sensor features
    # -----------------------------------------------------
    for sensor in df.columns:
        x_raw = df[sensor]
        x = x_raw.fillna(method="ffill").fillna(method="bfill").values  # safe fill
        
        # Missingness features
        features[f"{sensor}_missing_fraction"] = x_raw.isna().mean()
        features[f"{sensor}_missing_count"] = x_raw.isna().sum()
        
        # Longest missing streak
        streaks = x_raw.isna().astype(int)
        longest_streak = longest_run(streaks.values)
        features[f"{sensor}_missing_longest_streak"] = longest_streak
        
        # Basic stats
        features[f"{sensor}_mean"] = np.mean(x)
        features[f"{sensor}_std"] = np.std(x)
        features[f"{sensor}_min"] = np.min(x)
        features[f"{sensor}_max"] = np.max(x)
        features[f"{sensor}_iqr"] = np.percentile(x, 75) - np.percentile(x, 25)
        features[f"{sensor}_cov"] = np.std(x) / (np.mean(x) + 1e-6)
        
        # Temporal dynamics
        features.update({
            f"{sensor}_{k}": v
            for k, v in derivative_features(x).items()
        })
        
        # Out-of-range
        min_val, max_val = ranges.get(sensor, (np.min(x), np.max(x)))
        features[f"{sensor}_oor_fraction"] = out_of_range_fraction(x, min_val, max_val)
        
        # Extreme jumps
        threshold = jump_thresholds.get(sensor, np.std(x) * 3)
        features[f"{sensor}_extreme_jump_fraction"] = extreme_jump_fraction(x, threshold)
        
        # Spectral entropy
        features[f"{sensor}_spectral_entropy"] = spectral_entropy(x)
        
        # Drift detection
        if baseline_stats and sensor in baseline_stats:
            base_mean, base_std = baseline_stats[sensor]
            features[f"{sensor}_mean_drift"] = abs(np.mean(x) - base_mean)
            features[f"{sensor}_std_drift"] = abs(np.std(x) - base_std)
    
    # -----------------------------------------------------
    # 2. Composite trust indicators
    # -----------------------------------------------------
    features["num_sensors_high_oor"] = sum(
        features[f"{s}_oor_fraction"] > 0.05 for s in df.columns
    )
    
    features["num_sensors_high_jump"] = sum(
        features[f"{s}_extreme_jump_fraction"] > 0.05 for s in df.columns
    )
    
    features["num_sensors_high_drift"] = sum(
        features.get(f"{s}_mean_drift", 0) > 0.1 for s in df.columns
    )
    
    return pd.DataFrame([features])


In [ ]:
# Load data
def load_data():
    train = pd.read_csv(TRAIN_PATH)
    val = pd.read_csv(VAL_PATH)
    test = pd.read_csv(TEST_PATH)

    #need to drop timestamp, PW_0.5h col
    train = train.drop(columns=['timestamp', 'PW_0.5h'])
    val = val.drop(columns=['timestamp', 'PW_0.5h'])
    test = test.drop(columns=['timestamp', 'PW_0.5h'])

    return train, val, test


In [11]:
#build feature matrix
def build_feature_matrix(rw_list, ranges, jump_thresholds, baseline_stats):
    """
    rw_list: list of RW dataframes
    returns: feature matrix (DataFrame)
    """
    feature_rows = []
    for rw in rw_list:
        feats = extract_features(rw, ranges, jump_thresholds, baseline_stats)
        feature_rows.append(feats)
    return pd.concat(feature_rows, ignore_index=True)

In [12]:
#train
def train_isolation_forest(feature_matrix, contamination=0.05):
    """
    Train Isolation Forest on extracted features.
    """
    # Scale features
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(feature_matrix)

    # Train model
    iso = IsolationForest(
        n_estimators=300,
        contamination=contamination,
        max_samples='auto',
        random_state=42,
        bootstrap=False
    )
    iso.fit(X_scaled)

    # Save model + scaler
    joblib.dump(iso, MODEL_PATH)
    joblib.dump(scaler, SCALER_PATH)

    print("Isolation Forest training complete.")
    print(f"Model saved to: {MODEL_PATH}")
    print(f"Scaler saved to: {SCALER_PATH}")

    return iso, scaler

In [13]:
#Load model for runtime use
def load_iso_model():
    iso = joblib.load(MODEL_PATH)
    scaler = joblib.load(SCALER_PATH)
    return iso, scaler

In [ ]:
#compute trust score for a single RW
def compute_trust_score(rw, ranges, jump_thresholds, baseline_stats):
    """
    Returns trust score and anomaly flag for a single RW.
    """
    iso, scaler = load_iso_model()

    feats = extract_features(rw, ranges, jump_thresholds, baseline_stats)
    X_scaled = scaler.transform(feats)

    #Trust score: higher = more normal, lower = more anomalous [0, 1]
    trust_score = iso.score_samples(X_scaled)[0]*-1 
    # Anomaly flag (0 = normal, 1 = anomaly)
    anomaly = iso.predict(X_scaled)[0]== -1

    return trust_score, anomaly

In [ ]:
#batch scoring for multiple RWs
def score_many_rws(rw_list, ranges, jump_thresholds, baseline_stats):
    iso, scaler = load_iso_model()

    results = []
    for rw in rw_list:
        feats = extract_features(rw, ranges, jump_thresholds, baseline_stats)
        X_scaled = scaler.transform(feats)

        trust_score = iso.score_samples(X_scaled)[0]*-1
        anomaly = iso.predict(X_scaled)[0]

        results.append({
            "trust_score": trust_score,
            "anomaly": anomaly
        })

    return pd.DataFrame(results)

In [ ]:
# MAIN WORKFLOW

def main():
    #TD: fine tune ranges, jump thresholds, and baseline stats based on domain knowledge
    ranges = {
        "CMS air pressure": (0, 9.5),
        "Concentrazione ossigeno base": (5, 500),
        "Nitrogen pressure": (0, 9),
        "Oxygen over threshold": (0, 1)
    }

    jump_thresholds = {
        "CMS air pressure": 5,
        "Concentrazione ossigeno base": 1,
        "Nitrogen pressure": 0.2,
        "Oxygen over threshold": 0.1
    }

    baseline_stats = {
        "CMS air pressure": (1000, 10),
        "Concentrazione ossigeno base": (21, 1),
        "Nitrogen pressure": (2.0, 0.2),
        "Oxygen over threshold": (-0.045, 22.5)
    }

    train, val, test = load_data()


    #split data into RWs of 30 minutes
    train_rw = split_into_rws(train, 30)

    feature_matrix = build_feature_matrix(train_rw, ranges, jump_thresholds, baseline_stats)
    iso, scaler = train_isolation_forest(feature_matrix)
    
    

In [17]:
main()

Isolation Forest training complete.
Model saved to: iso_forest_nitrogen.pkl
Scaler saved to: iso_scaler.pkl
